# DAY 3: Supervised Fine-Tuning (SFT) & Chat Templates

## 🎯 Learning Objectives
By the end of this day, you will be able to explain and demonstrate:
1. **Pretraining vs. Supervised Fine-Tuning (SFT)**: Transitioning from next-token completion on raw web text to following structured instructions.
2. **Chat Templates & Special Tokens**: Understanding roles (`system`, `user`, `assistant`) and special delimiters (`<|im_start|>`, `<|im_end|>`).
3. **Assistant Response Loss Masking**: Why we set prompt labels to `-100` (`ignore_index`) so backpropagation only computes gradients on the target response.
4. **Causal Language Modeling Objective**: The mathematics of auto-regressive cross-entropy loss $\mathcal{L} = -\sum_{t=1}^{T} \log P(x_t \mid x_{<t})$.
5. **TRL `SFTTrainer`**: Using modern Hugging Face TRL utilities to streamline multi-turn and instruction tuning.

## 📐 The Causal LM Loss & Loss Masking

Given a sequence of token IDs $X = (x_1, x_2, \dots, x_N)$ where the prompt occupies indices $1 \dots K$ and the target response occupies $K+1 \dots N$:

$$
\mathcal{L}_{\text{full sequence}} = -\sum_{t=1}^{N} \log P(x_t \mid x_{<t}; \theta)
$$

### Why Loss Masking Matters:
If we compute loss over the entire sequence (including system prompt and user input), the model wastes gradient updates learning to predict the user prompt (which varies randomly across tasks). In **instruction tuning**, we only want the model to learn the probability distribution of the **assistant completion** conditioned on the user prompt:

$$
\mathcal{L}_{\text{masked SFT}} = -\sum_{t=K+1}^{N} \log P(x_t \mid x_{<t}; \theta)
$$

In PyTorch `nn.CrossEntropyLoss`, setting `labels[1:K] = -100` causes the loss function to ignore those positions entirely during the forward and backward passes.

In [ ]:
# !pip install -q transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Loaded tokenizer: {MODEL_ID}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

### Step 1: Inspect and Apply Chat Template

In [ ]:
sample_conversation = [
    {
        "role": "system",
        "content": "You are an expert entity extraction system. Extract structured order information into JSON."
    },
    {
        "role": "user",
        "content": "John ordered 3 laptops for $2400 and wants delivery on Friday."
    },
    {
        "role": "assistant",
        "content": '{"customer": "John", "quantity": 3, "product": "laptops", "amount": 2400.0, "delivery_day": "Friday"}'
    }
]

# 1. Render formatted text string using Jinja chat template
rendered_text = tokenizer.apply_chat_template(sample_conversation, tokenize=False)
print("=== RENDERED CHAT TEMPLATE STRING ===")
print(rendered_text)
print("=" * 45)

### Step 2: Understand Tokenization & Loss Masking (`-100` labels)

In [ ]:
# Tokenize full conversation
tokenized = tokenizer(rendered_text, return_tensors="pt")
input_ids = tokenized["input_ids"][0]
labels = input_ids.clone()

# Find assistant start marker
prompt_only = sample_conversation[:2]
prompt_text = tokenizer.apply_chat_template(prompt_only, tokenize=False, add_generation_prompt=True)
prompt_len = len(tokenizer(prompt_text)["input_ids"])

# Mask prompt tokens by setting them to -100
labels[:prompt_len] = -100

print(f"Total Sequence Length : {len(input_ids)} tokens")
print(f"Prompt Tokens (Masked): {prompt_len} tokens (Label = -100)")
print(f"Target Completion     : {len(input_ids) - prompt_len} tokens (Gradients active)")

# Inspect unmasked tokens
active_token_ids = labels[labels != -100]
print("\nTokens with active gradient loss:")
print(tokenizer.decode(active_token_ids))

### Step 3: Prepare SFT Dataset with Hugging Face Datasets

In [ ]:
from datasets import Dataset

# Sample dataset mini-batch
raw_records = [
    {
        "messages": [
            {"role": "system", "content": "Extract order info into JSON."},
            {"role": "user", "content": "Alice bought 2 headphones for $300, delivery Tuesday."},
            {"role": "assistant", "content": '{"customer": "Alice", "quantity": 2, "product": "headphones", "amount": 300.0, "delivery_day": "Tuesday"}'}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "Extract order info into JSON."},
            {"role": "user", "content": "Bob ordered 1 standing desk for $450, deliver tomorrow."},
            {"role": "assistant", "content": '{"customer": "Bob", "quantity": 1, "product": "standing desk", "amount": 450.0, "delivery_day": "tomorrow"}'}
        ]
    }
]

hf_dataset = Dataset.from_list(raw_records)
print(f"HF Dataset Created: {hf_dataset}")
print("Sample Entry:", hf_dataset[0])

---
## 🎓 Day 3 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **Chat Templates**: Jinja-based templating system mapping structured message lists (`role`, `content`) into model-specific token streams.
2. **Special Tokens**: Control tokens like `<|im_start|>`, `<|im_end|>`, `<|endoftext|>` that mark boundaries between turns.
3. **Loss Masking (`ignore_index = -100`)**: Disabling gradient computation on prompts so model capacity focuses exclusively on generating correct answers.
4. **Causal Attention Mask**: Lower-triangular attention matrix ensuring token $t$ can only attend to tokens $\le t$.
5. **Overfitting vs Underfitting in SFT**: Monitoring train vs validation loss curves to identify when adapter weights over-memorize training examples.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *How does Supervised Fine-Tuning differ from Pretraining?*
   - **Answer**: Pretraining uses unsupervised next-token prediction across trillions of diverse web tokens to learn general language, reasoning, and world knowledge. SFT tunes model weights on curated (prompt, response) pairs to shape outputs into helpful, structured, or aligned responses.
2. **Q2 (Intermediate)**: *Why must we format training data using the model's exact native chat template?*
   - **Answer**: Pretrained instruction models have learned semantic meanings for specific control tokens (e.g. Qwen's `<|im_start|>assistant` vs Llama's `<|start_header_id|>assistant<|end_header_id|>`). Deviating causes distribution shifts and degrades instruction-following capabilities.
3. **Q3 (Intermediate)**: *What happens if you do NOT mask prompt loss during SFT?*
   - **Answer**: The model splits its learning capacity between predicting user prompt tokens and predicting target outputs. If user prompts are noisy or long, the model may memorize prompt phrasing rather than mastering the output task.
4. **Q4 (Advanced)**: *How does TRL's `DataCollatorForCompletionOnlyLM` work under the hood?*
   - **Answer**: It tokenizes the full sequence and locates the sub-sequence corresponding to the assistant delimiter (e.g. `<|im_start|>assistant\n`). All token positions before that index are set to `-100` in the `labels` tensor.
5. **Q5 (Advanced)**: *What learning rate schedule is best for SFT and why?*
   - **Answer**: A **Cosine Annealing schedule with a linear warmup** (e.g., 3-5% warmup ratio). Warmup prevents large destructive gradient updates in early steps when adapter weights are initialized, and cosine decay smoothly anneals learning rate to zero for fine convergence.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, how did you verify which tokens received active gradients vs. which were masked with `-100`?*
2. *What is `add_generation_prompt=True` and when should it be used during inference vs training?*
3. *How does `tokenizer.pad_token = tokenizer.eos_token` affect batch padding?*

### 🏆 Day 3 Hands-On Challenge
> **Challenge**: Write a custom Python function `mask_prompt_labels(input_ids, tokenizer, response_template="<|im_start|>assistant\n")` that automatically detects the response delimiter token sequence and returns the masked `labels` tensor.